# 🧠 Building Autoencoders with Keras


## 📋 Overview

I build an autoencoder — a neural network that learns to compress data down to a small representation and then reconstruct it, with no labels involved. I train it on MNIST digits, evaluate how well it reconstructs unseen images, fine-tune it by selectively unfreezing layers, and then push it further into a denoising task where it has to recover clean digits from noisy input.

Coming from a telecom/RF background, I keep coming back to one analogy: an autoencoder is a *learned codec*. The encoder is a source coder squeezing a signal into a compact representation, the bottleneck is the compressed bitstream, and the decoder is the receiver reconstructing the original signal from that compressed form. Everything I do in this notebook — training, fine-tuning, denoising — maps cleanly onto ideas I already know from signal processing.

**What I cover:**
- 📥 Preprocessing MNIST for a dense autoencoder
- 🏗️ Building an encoder → bottleneck → decoder with the Keras Functional API
- 🔄 Training on a reconstruction objective (target = input)
- 📊 Evaluating reconstruction quality visually
- 🧊 Fine-tuning by freezing/unfreezing layers
- 🧪 Denoising noisy digits
- 🎯 Practice: bottleneck size, L2 regularization, and latent space visualization


## 🧩 Theory

An autoencoder is two networks stitched together and trained end-to-end on a single objective: **reconstruct the input**.

$$
z = f_\theta(x), \qquad \hat{x} = g_\phi(z), \qquad \mathcal{L}(\theta, \phi) = \frac{1}{N}\sum_{i=1}^{N} \text{BCE}(x_i, \hat{x}_i)
$$

- $x$ — the original input (a flattened 28×28 MNIST image, so a 784-dim vector)
- $f_\theta$ — the **encoder**, compressing $x$ into a smaller latent code $z$
- $z$ — the **bottleneck**, the compressed representation (32 dimensions here — a 24.5× reduction from 784)
- $g_\phi$ — the **decoder**, reconstructing $\hat{x}$ from $z$
- $\mathcal{L}$ — binary cross-entropy between original pixels and reconstructed pixels, since pixel values are normalized to $[0, 1]$

There are no labels anywhere in this loss — the "label" for each image is the image itself. That's what makes this unsupervised.

### 📡 Telecom analogy

| Autoencoder piece | Signal processing equivalent |
|---|---|
| Encoder $f_\theta$ | Source coder — compress a signal to the fewest bits that preserve its structure (e.g. speech/image codecs) |
| Bottleneck $z$ | The compressed bitstream / channel-capacity-constrained representation |
| Decoder $g_\phi$ | Source decoder — reconstruct the signal from the compressed form |
| Reconstruction loss | Distortion metric (rate-distortion tradeoff: smaller bottleneck = higher potential distortion) |
| Denoising variant | A receiver recovering the clean signal from a noisy channel — conceptually close to a matched filter |

The smaller I make the bottleneck, the harder the encoder has to work to preserve what matters — exactly the rate-distortion tradeoff I'd reason about when picking a compression ratio for a telecom link.


## Part 1 — 📥 Data Preprocessing

I load MNIST, normalize pixel values to $[0, 1]$, and flatten each 28×28 image into a 784-dim vector. Normalization keeps gradients well-scaled during training (the neural-net equivalent of AGC — automatic gain control — keeping signal amplitude in a consistent range before further processing). Flattening is required because my encoder/decoder are `Dense` layers, which expect 1D vectors, not 2D images.


In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist

# Load the dataset
(x_train, _), (x_test, _) = mnist.load_data()

# Normalize the pixel values to [0, 1] — keeps gradients well-scaled, like AGC on an incoming signal
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

# Flatten the 28x28 images into 784-dim vectors — Dense layers expect 1D input
x_train = x_train.reshape((len(x_train), np.prod(x_train.shape[1:])))
x_test = x_test.reshape((len(x_test), np.prod(x_test.shape[1:])))

print(f"Train shape: {x_train.shape}, Test shape: {x_test.shape}")


**What just happened:**

| Step | Purpose |
|---|---|
| Load MNIST | 60,000 training / 10,000 test grayscale digit images |
| Normalize to $[0,1]$ | Faster, more stable convergence during training |
| Flatten to 784-dim | Match the input shape my Dense-based encoder expects |


## Part 2 — 🏗️ Building the Autoencoder

I wire up the encoder, bottleneck, and decoder with the Keras Functional API: 784 → 64 → **32** → 64 → 784. The 32-dim bottleneck is a 24.5× compression of the original 784-dim input — every image has to survive being squeezed through that narrow point.


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# Encoder
input_layer = Input(shape=(784,))
encoded = Dense(64, activation='relu')(input_layer)

# Bottleneck — the compressed representation
bottleneck = Dense(32, activation='relu')(encoded)

# Decoder
decoded = Dense(64, activation='relu')(bottleneck)
output_layer = Dense(784, activation='sigmoid')(decoded)

# Autoencoder model: input -> reconstruction
autoencoder = Model(input_layer, output_layer)

# Compile: Adam optimizer, binary crossentropy since pixels are in [0, 1]
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

autoencoder.summary()


**Architecture summary:**

| Layer | Role | Units | Activation |
|---|---|---|---|
| Input | Raw flattened image | 784 | — |
| Dense (encoder) | First compression step | 64 | ReLU |
| Dense (bottleneck) | Latent code $z$ | 32 | ReLU |
| Dense (decoder) | First expansion step | 64 | ReLU |
| Dense (output) | Reconstruction $\hat{x}$ | 784 | Sigmoid (matches $[0,1]$ pixel range) |


## Part 3 — 🔄 Training the Autoencoder

The target is the input itself — `x_train` appears on both sides of `fit()`. That's the entire trick behind unsupervised reconstruction learning.


In [ ]:
autoencoder.fit(
    x_train, x_train,
    epochs=25,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test, x_test)
)


## Part 4 — 📊 Evaluating Reconstruction Quality

I run the test set through the trained autoencoder and compare original vs. reconstructed digits side by side. This is a visual sanity check — if reconstructions look close to the originals, the bottleneck genuinely captured the structure of the digits rather than just memorizing pixels.


In [ ]:
!pip install matplotlib==3.9.2


In [ ]:
import matplotlib.pyplot as plt

# Predict on the test set
reconstructed = autoencoder.predict(x_test)

# Visualize original vs reconstructed
n = 10  # number of digits to display
plt.figure(figsize=(20, 4))

for i in range(n):
    # Original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Reconstruction
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(reconstructed[i].reshape(28, 28))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

plt.show()


**Reading the result:** top row = originals, bottom row = reconstructions. Blurring or loss of fine detail here is the visual signature of the rate-distortion tradeoff — the 32-dim bottleneck can't preserve everything, so it prioritizes the structure that matters most for reconstruction.


## Part 5 — 🧊 Fine-Tuning by Freezing/Unfreezing Layers

I freeze every layer, confirm the frozen state, then unfreeze just the last four layers (essentially the decoder) and retrain. This mirrors a transfer-learning workflow: lock in what's already learned, then only recalibrate the part of the network closest to the output — like re-tuning only the final amplifier stage of a transmitter chain while leaving the upstream stages fixed.


In [ ]:
# Freeze all layers of the autoencoder
for layer in autoencoder.layers:
    layer.trainable = False


In [ ]:
# Check trainable status of each layer
for i, layer in enumerate(autoencoder.layers):
    print(f"Layer {i}: {layer.name}, Trainable = {layer.trainable}")


In [ ]:
# Unfreeze the last four layers (decoder side)
for layer in autoencoder.layers[-4:]:
    layer.trainable = True

# Recompile — required whenever trainable flags change
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

# Train again, fewer epochs since most of the network is already tuned
autoencoder.fit(x_train, x_train,
                epochs=10,
                batch_size=256,
                shuffle=True,
                validation_data=(x_test, x_test))


## Part 6 — 🧪 Denoising Images with the Autoencoder

I corrupt the input with Gaussian noise and train the autoencoder to map noisy input → clean target. This is the same reconstruction objective as before, just with a noisy $x$ on the input side:

$$
\tilde{x} = x + \eta, \qquad \eta \sim \mathcal{N}(0, \sigma^2), \qquad \tilde{x} = \text{clip}(\tilde{x}, 0, 1)
$$

Conceptually this is close to a receiver recovering a clean signal from a noisy channel — a learned matched filter, except the "filter" is whatever the encoder/decoder pair discovers during training rather than something I derive analytically.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Add Gaussian noise to the data
noise_factor = 0.5
x_train_noisy = x_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy = x_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)
x_train_noisy = np.clip(x_train_noisy, 0., 1.)
x_test_noisy = np.clip(x_test_noisy, 0., 1.)

# Train on noisy input -> clean target
autoencoder.fit(
    x_train_noisy, x_train,
    epochs=20,
    batch_size=512,
    shuffle=True,
    validation_data=(x_test_noisy, x_test)
)

# Denoise the test images
reconstructed_noisy = autoencoder.predict(x_test_noisy)

# Visualize noisy / denoised / original
n = 10
plt.figure(figsize=(20, 6))
for i in range(n):
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_test_noisy[i].reshape(28, 28))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(reconstructed_noisy[i].reshape(28, 28))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(x_test[i].reshape(28, 28))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

plt.show()


**Reading the result:** top = noisy input, middle = denoised output, bottom = ground truth. The closer the middle row is to the bottom row, the better the network learned to separate signal from noise rather than just memorizing clean digits.


## 🎯 Practice: Comparing Bottleneck Sizes

**Objective:** understand how the bottleneck size trades off compression against reconstruction quality — the same rate-distortion tradeoff I'd reason about when sizing a compressed telecom link.

I train three otherwise-identical autoencoders with bottleneck sizes 16, 32, and 64, then compare test loss.


In [ ]:
# Define and train three autoencoders with varying bottleneck sizes
bottleneck_sizes = [16, 32, 64]
autoencoders = []

for size in bottleneck_sizes:
    # Encoder
    input_layer = Input(shape=(784,))
    encoded = Dense(64, activation='relu')(input_layer)
    bottleneck = Dense(size, activation='relu')(encoded)

    # Decoder
    decoded = Dense(64, activation='relu')(bottleneck)
    output_layer = Dense(784, activation='sigmoid')(decoded)

    # Autoencoder model
    ae = Model(input_layer, output_layer)
    ae.compile(optimizer='adam', loss='binary_crossentropy')
    ae.fit(
        x_train,
        x_train,
        epochs=20,
        batch_size=256,
        shuffle=True,
        validation_data=(x_test, x_test)
    )
    autoencoders.append(ae)

# Evaluate and compare
for i, size in enumerate(bottleneck_sizes):
    loss = autoencoders[i].evaluate(x_test, x_test)
    print(f'Bottleneck size {size} - Test loss: {loss}')


**What I expect:** loss should generally decrease as bottleneck size grows (64 > 32 > 16 in capacity, so lower distortion) — with diminishing returns once the bottleneck is large enough to stop being the binding constraint. Same shape as a rate-distortion curve: more "bits" (dimensions) buys better fidelity until the input's true information content is reached.


## ⚙️ Practice: Adding L2 Regularization

**Objective:** see how penalizing large weights affects reconstruction. L2 regularization adds a penalty term proportional to the sum of squared weights:

$$
\mathcal{L}_{\text{reg}} = \mathcal{L}_{\text{BCE}} + \lambda \sum_i w_i^2
$$

This discourages any single weight from growing too large — the neural-net analogue of limiting transmit power to avoid amplifier saturation and distortion. Too much regularization (large $\lambda$) under-fits, same as clamping power so hard the signal itself gets suppressed.


In [ ]:
from tensorflow.keras.regularizers import l2

# Encoder with L2 regularization
input_layer = Input(shape=(784,))
encoded = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(input_layer)
bottleneck = Dense(32, activation='relu', kernel_regularizer=l2(0.01))(encoded)

# Decoder with L2 regularization
decoded = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(bottleneck)
output_layer = Dense(784, activation='sigmoid', kernel_regularizer=l2(0.01))(decoded)

# Autoencoder with L2 regularization
autoencoder_regularized = Model(input_layer, output_layer)
autoencoder_regularized.compile(optimizer='adam', loss='binary_crossentropy')

# Train
autoencoder_regularized.fit(x_train, x_train, epochs=50, batch_size=256, shuffle=True, validation_data=(x_test, x_test))

# Evaluate
loss = autoencoder_regularized.evaluate(x_test, x_test)
print(f'Regularized Autoencoder - Test loss: {loss}')


## 🔍 Practice: Visualizing the Latent Space

**Objective:** look at what the bottleneck actually learned. I extract just the encoder half as its own model, run the test set through it, and scatter-plot the first two latent dimensions.

This is conceptually the same move as PCA or t-SNE from earlier modules — projecting high-dimensional data down to something plottable — except the encoder's projection is non-linear and learned specifically to support reconstruction, not just to preserve variance.


In [ ]:
import matplotlib.pyplot as plt

# Extract the encoder half of the (first) autoencoder
encoder_model = Model(input_layer, bottleneck)

# Encode the test set
encoded_imgs = encoder_model.predict(x_test)

# Visualize the first two latent dimensions
plt.figure(figsize=(10, 8))
plt.scatter(encoded_imgs[:, 0], encoded_imgs[:, 1], c='blue', alpha=0.5)
plt.title('Encoded Features - First Two Dimensions')
plt.xlabel('Encoded Feature 1')
plt.ylabel('Encoded Feature 2')
plt.show()


## 📊 Summary

| Concept | What I did | Why it matters |
|---|---|---|
| 📥 Preprocessing | Normalized pixels to $[0,1]$, flattened images to 784-dim | Matches Dense layer input requirements, stabilizes gradients |
| 🏗️ Functional API | Wired encoder → bottleneck (32) → decoder | Reusable, graph-based way to define non-sequential architectures |
| 🔄 Reconstruction training | `fit(x_train, x_train, ...)` | Core trick of unsupervised autoencoding — no labels needed |
| 📊 Evaluation | Visual comparison of original vs. reconstructed digits | Sanity-checks that the bottleneck preserved meaningful structure |
| 🧊 Fine-tuning | Froze all layers, unfroze last 4, retrained | Mirrors transfer-learning workflows — recalibrate only what's needed |
| 🧪 Denoising | Trained noisy → clean instead of clean → clean | Extends reconstruction to a practical noise-removal task |
| 🎯 Bottleneck size | Compared loss across 16 / 32 / 64 dims | Rate-distortion tradeoff — smaller bottleneck, higher potential loss |
| ⚙️ L2 regularization | Added `kernel_regularizer=l2(0.01)` to every layer | Penalizes large weights, analogous to limiting transmit power |
| 🔍 Latent space | Plotted first 2 dims of the encoder's output | Non-linear analogue of PCA — a learned, task-specific projection |

**Telecom throughline:** every section of this notebook mapped onto something from signal processing — codecs (encoder/decoder), rate-distortion (bottleneck size), power limiting (L2 regularization), and matched filtering (denoising). Autoencoders are, in a real sense, learned compression and noise-recovery systems.


## 🧪 Sandbox

Space to keep experimenting beyond the practice exercises:

- Swap the Dense encoder/decoder for `Conv2D` / `Conv2DTranspose` and compare reconstruction quality on the same MNIST data
- Try a much smaller bottleneck (e.g. 2 dimensions) so the latent scatter plot in Part 9 becomes fully interpretable
- Push `noise_factor` higher (0.7–1.0) and see where the denoising autoencoder starts to fail
- Apply this same encoder/decoder pattern to a non-image signal — e.g. a synthetic RF spectrogram or time-series — and see if the compression/reconstruction story still holds
- Try a variational autoencoder (VAE) next to compare a probabilistic bottleneck against this deterministic one


In [ ]:
# 🧪 Sandbox — experiment here
